<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/answer/Chapter06_attention_head_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 06 — Attention Head — the core operation**

> ✅ Answer notebook — full working code.

## Goal — the most important chapter

A single **causal self-attention head**. Each token produces a **query** (what am I looking for?),
a **key** (what do I offer?) and a **value** (what I'll pass on). A token attends to earlier tokens
by comparing its query to their keys, then takes a weighted sum of their values.

Shapes: from `x: (B, T, C)` we make `q, k, v: (B, T, head_size)`. The attention matrix is
`(B, T, T)`. **Causal masking** sets future positions to `-inf` so a token can only see the past.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## Step 1 · Q, K, V projections and the score matrix

In [ ]:
C, head_size, block_size = 32, 16, 8

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

x = torch.randn(4, block_size, C)              # (B, T, C)
k = key(x)                                     # (B, T, head_size)
q = query(x)
v = value(x)

scores = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T), scaled
print("scores:", scores.shape)

## Step 2 · Causal mask + softmax + weighted sum

We mask the upper triangle (future) to `-inf`, softmax over the last dim to get weights that sum to
1, then multiply by `v`.

In [ ]:
tril = torch.tril(torch.ones(block_size, block_size))
scores = scores.masked_fill(tril == 0, float("-inf"))   # block future
weights = F.softmax(scores, dim=-1)                     # (B, T, T)
out = weights @ v                                       # (B, T, head_size)
print("attention weights row 0 (token 0 sees only itself):")
print(weights[0, 0].round(decimals=2))
print("out:", out.shape)

## Step 3 · Package as a Head module

In [ ]:
class Head(nn.Module):
    def __init__(self, C, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.scale = head_size ** -0.5
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = q @ k.transpose(-2, -1) * self.scale
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        return att @ v

h = Head(32, 16, 8)
print("Head output:", h(torch.randn(4, 8, 32)).shape)    # (4, 8, 16)

✅ You built attention. Next: run several heads in parallel.